In [52]:
#Retrieve the dataset with test summaries by human and T5
%store -r test_paired_summaries
%store -r t5_test_sumaries

%store -r val_paired_summaries
%store -r t5_val_sumaries


In [2]:
import sys

!{sys.executable} -m pip install -q rouge-score bert-score scikit-learn
!{sys.executable} -m pip install bert-score
!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu


In [3]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np

from bert_score import score as bertscore

import torch

from transformers import AutoModel


/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [9]:
#rouge and cosine similarity methods (created with the help of ChatGPT)
def compute_rouge(text1, text2):
    """
    Compute ROUGE between two texts.
    Returns F1 scores for ROUGE-1, ROUGE-2, ROUGE-L.
    """
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )
    # Note: scorer.score(reference, prediction)
    scores = scorer.score(text2, text1)

    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def compute_cosine_similarity(text1, text2):
    """
    Compute cosine similarity between two texts using TF-IDF vectors.
    This avoids loading any transformer model and is stable in most environments.
    """
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text1, text2])  # shape (2, vocab_size)

    cos_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])  # (1,1)
    return float(cos_sim[0, 0])

def compute_bertscore(text1: str, text2: str):
    """
    Compute BERTScore (Precision, Recall, F1) between two texts.
    text1 = candidate
    text2 = reference
    """
    P, R, F1 = bertscore(
        cands=[text1],
        refs=[text2],
        lang="en"
    )
    return {
        "precision": float(P[0]),
        "recall": float(R[0]),
        "f1": float(F1[0]),
    }


In [12]:
import warnings
warnings.filterwarnings("ignore", message="Some weights of RobertaModel")

In [13]:
#For Testing
text_a = "How old are you going to be tomorrow?"
text_b = "What will your age be on the day that follows today"

print(compute_rouge(text_a, text_b))
print(compute_cosine_similarity(text_a, text_b))
print(compute_bertscore(text_a, text_b))

{'rouge1': 0.10526315789473685, 'rouge2': 0.0, 'rougeL': 0.10526315789473685}
0.05700655341923561


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'precision': 0.8961870670318604, 'recall': 0.8896476030349731, 'f1': 0.8929053544998169}


In [26]:
#For looking at data
print(len(test_paired_summaries))
print(len(t5_test_sumaries))

print(list(test_paired_summaries.keys())[20])
print(list(t5_test_sumaries.keys())[20])

print(list(test_paired_summaries.values())[20][1])
print(list(t5_test_sumaries.values())[20])


141
141
MacBeth-Act I-Scene III
MacBeth-Act I-Scene III
The witches prophesy that Macbeth shall be king and Banquo shall be father of kings. . . . Ross and Angus tell Macbeth he has been given the title of Thane of Cawdor. . . . Macbeth muses on the possibility of killing the King in order to be king.
sailor's wife had chestnuts in her lap, and mounch'd, and mounch'd . ronyon cries: "Give me," "Aroint thee, witch!" . he says, "i’ll give thee a wind."


## Test Data set

In [36]:
#ROUGE
#Create a dictionary which will have key as scene/sonnet name and value of rougediff
#Value will be tuple which contains (rouge1, rouge2. rougeL)
rouge_test_summary_eval=dict()

for k, v in test_paired_summaries.items():
    human_summary=v[1]
    t5_summary=t5_test_sumaries[k]

    rouge_diff = compute_rouge(t5_summary, human_summary)

    rouge_test_summary_eval[k]=(rouge_diff["rouge1"], rouge_diff["rouge2"], rouge_diff["rougeL"])

In [37]:
#Cosine Sim
#Create a dictionary which will have key as scene/sonnet name and value of cosine similarity
cosine_test_summary_eval=dict()

for k, v in test_paired_summaries.items():
    human_summary=v[1]
    t5_summary=t5_test_sumaries[k]

    cosine_diff = compute_cosine_similarity(t5_summary, human_summary)

    cosine_test_summary_eval[k]=cosine_diff

In [44]:
#BertScore
#Create a dictionary which will have key as scene/sonnet name and value of BertScore
#Value will be tuple which contains (precision, recall. F1)
bertscore_test_summary_eval=dict()
i=0

for k, v in test_paired_summaries.items():
    human_summary=v[1]
    t5_summary=t5_test_sumaries[k]

    bertscore_diff = compute_bertscore(t5_summary, human_summary)
    bertscore_test_summary_eval[k]=(bertscore_diff["precision"], bertscore_diff["recall"], bertscore_diff["f1"])
    
    #For tracking progress
    if i%10==0:
        print(i)
    i+=1


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


0


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

10


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

...

120


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

130


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

140


In [51]:
%store rouge_test_summary_eval
%store cosine_test_summary_eval
%store bertscore_test_summary_eval

Stored 'rouge_test_summary_eval' (dict)
Stored 'cosine_test_summary_eval' (dict)
Stored 'bertscore_test_summary_eval' (dict)


## Validation Data Set

In [53]:
#ROUGE
#Create a dictionary which will have key as scene/sonnet name and value of rougediff
#Value will be tuple which contains (rouge1, rouge2. rougeL)
rouge_val_summary_eval=dict()

for k, v in val_paired_summaries.items():
    human_summary=v[1]
    t5_summary=t5_val_sumaries[k]

    rouge_diff = compute_rouge(t5_summary, human_summary)

    rouge_val_summary_eval[k]=(rouge_diff["rouge1"], rouge_diff["rouge2"], rouge_diff["rougeL"])

In [56]:
#Cosine Sim
#Create a dictionary which will have key as scene/sonnet name and value of cosine similarity
cosine_val_summary_eval=dict()

for k, v in val_paired_summaries.items():
    human_summary=v[1]
    t5_summary=t5_val_sumaries[k]

    cosine_diff = compute_cosine_similarity(t5_summary, human_summary)

    cosine_val_summary_eval[k]=cosine_diff

In [59]:
#BertScore
#Create a dictionary which will have key as scene/sonnet name and value of BertScore
#Value will be tuple which contains (precision, recall. F1)
bertscore_val_summary_eval=dict()
i=0

for k, v in val_paired_summaries.items():
    human_summary=v[1]
    t5_summary=t5_val_sumaries[k]

    bertscore_diff = compute_bertscore(t5_summary, human_summary)
    bertscore_val_summary_eval[k]=(bertscore_diff["precision"], bertscore_diff["recall"], bertscore_diff["f1"])
    
    #For tracking progress
    if i%10==0:
        print(i)
    i+=1


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


0


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

10


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

...

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

120


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

130


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

In [58]:
#For checking
print(len(rouge_val_summary_eval))
print(rouge_val_summary_eval)
print()

print(len(cosine_val_summary_eval))
print(cosine_val_summary_eval)
print()

print(len(bertscore_val_summary_eval))
print(bertscore_val_summary_eval)

136
{'Hamlet-Act I-Scene I': (0.1111111111111111, 0.0, 0.05555555555555555), 'Hamlet-Act I-Scene II': (0.06557377049180328, 0.0, 0.03278688524590164), 'Hamlet-Act I-Scene III': (0.03508771929824561, 0.0, 0.03508771929824561), 'Hamlet-Act I-Scene IV': (0.06451612903225806, 0.0, 0.06451612903225806), 'Hamlet-Act I-Scene V': (0.0625, 0.0, 0.0625), 'Hamlet-Act II-Scene I': (0.1739130434782609, 0.0, 0.13043478260869568), 'Hamlet-Act II-Scene II': (0.0963855421686747, 0.02469135802469136, 0.0963855421686747), 'Hamlet-Act III-Scene I': (0.09803921568627451, 0.0, 0.058823529411764705), 'Hamlet-Act III-Scene II': (0.1090909090909091, 0.0, 0.1090909090909091), 'Hamlet-Act III-Scene III': (0.0392156862745098, 0.0, 0.0392156862745098), 'Hamlet-Act III-Scene IV': (0.09302325581395349, 0.0, 0.09302325581395349), 'Hamlet-Act IV-Scene I': (0.14814814814814817, 0.07692307692307693, 0.14814814814814817), 'Hamlet-Act IV-Scene II': (0.043478260869565216, 0.0, 0.043478260869565216), 'Hamlet-Act IV-Scene II

In [60]:
%store rouge_val_summary_eval
%store cosine_val_summary_eval
%store bertscore_val_summary_eval

Stored 'rouge_val_summary_eval' (dict)
Stored 'cosine_val_summary_eval' (dict)
Stored 'bertscore_val_summary_eval' (dict)
